In [1]:
import os
import re
import numpy as np
import faiss
import torch
from tqdm import tqdm
from collections import Counter
from sentence_transformers import SentenceTransformer
from openai import OpenAI


##############################################
# 0. Ollama / Qwen Client
##############################################

# Ollama exposes an OpenAI-compatible API locally.
# Make sure Ollama is running:
#
#     ollama serve
#
# And that your Qwen model is installed:
#
#     ollama pull qwen3:8b

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # Required by the OpenAI client, but Ollama ignores it
)

QWEN_MODEL ="gpt-oss:20B"


##############################################
# 1. Semantic Processor
##############################################

def semantic_puml_processor(text):
    """
    Extracts meaningful tokens from PlantUML:
    - Design pattern roles
    - Class/interface names
    - Relationships
    """

    tokens = []

    # Extract DP roles
    roles = re.findall(
        r"'\s*DP role:\s*(\w+)",
        text,
        re.IGNORECASE
    )

    tokens.extend([
        f"role_{r.lower()}"
        for r in roles
    ])

    # Extract class/interface names
    classes = re.findall(
        r"(?:class|interface)\s+([\w\.]+)",
        text
    )

    for c in classes:
        tokens.append(
            c.split('.')[-1].lower()
        )

    # Map relationships
    rel_map = {
        r"<\|--": "inheritance",
        r"<\|..": "implementation",
        r"-->": "association",
        r"\*--": "composition"
    }

    for symbol, word in rel_map.items():
        if re.search(symbol, text):
            tokens.append(word)

    return " ".join(tokens)


##############################################
# 2. Document Loading (NO CHUNKING)
##############################################

def load_and_process_docs(folder_path):
    """
    Loads full PlantUML files (no chunking).
    """

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(
                os.path.join(folder_path, file),
                "r",
                encoding="utf-8"
            ) as f:

                raw_content = f.read()
                semantic_content = semantic_puml_processor(raw_content)

                if semantic_content.strip():
                    documents.append({
                        "id": file,
                        "content": semantic_content,
                        "raw": raw_content
                    })

    return documents


##############################################
# 3. Embedding Model
##############################################

class EmbeddingModel:

    def __init__(
        self,
        model_name="BAAI/bge-base-en"
    ):

        self.device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

        self.model = SentenceTransformer(
            model_name
        ).to(self.device)

    def encode(
        self,
        texts,
        batch_size=16
    ):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )


##############################################
# 4. Vector Store (FAISS)
##############################################

class VectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatIP(
            embedding_dim
        )

        self.metadata = []

    def add_embeddings(
        self,
        embeddings,
        metadatas
    ):

        self.index.add(embeddings)
        self.metadata.extend(metadatas)

    def search(
        self,
        query_embedding,
        top_k=5
    ):

        distances, indices = self.index.search(
            query_embedding,
            top_k
        )

        results = []

        for i, idx in enumerate(indices[0]):

            if idx != -1:

                res = self.metadata[idx].copy()
                res["score"] = distances[0][i]

                results.append(res)

        return results


##############################################
# 5. Qwen Reasoning via Ollama
##############################################

def llm_reasoning(query_raw, retrieved_docs):
    """
    Uses local gpt-oss:20B through Ollama to infer Singleton or nons if no Singleton is found.
    """

    examples = "\n\n".join(
        [
            f"Example ({doc['id']}):\n{doc['raw']}"
            for doc in retrieved_docs
        ]
    )

    prompt = f"""
    Your task is to identify exactly ONE GoF design pattern.
    Identify only the design pattern Singleton used in the following PlantUML diagram. 
    If no Singleton is found, respond with "nons".

QUERY:
{query_raw}

SIMILAR EXAMPLES:
{examples}


Answer ONLY with the Singleton or nons.


"""

    response = client.chat.completions.create(
        model=QWEN_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an expert in GoF software design patterns"
                
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()


##############################################
# 6. Build RAG Index
##############################################

def build_rag_index(folder_path):

    print("Loading PlantUML files...")

    docs = load_and_process_docs(folder_path)

    if not docs:
        raise ValueError(
            f"No PlantUML files found in: {folder_path}"
        )

    texts = [
        doc["content"]
        for doc in docs
    ]

    embedder = EmbeddingModel()

    embeddings = embedder.encode(texts)

    vector_store = VectorStore(
        embedding_dim=embeddings.shape[1]
    )

    vector_store.add_embeddings(
        np.array(embeddings),
        docs
    )

    return vector_store, embedder


##############################################
# 7. Pattern Detection
##############################################

def detect_pattern(
    test_doc,
    vector_store,
    embedder,
    top_k=5,
    use_llm=True
):

    query_text = semantic_puml_processor(
        test_doc["content"]
    )

    query_embedding = embedder.encode(
        [query_text]
    )

    results = vector_store.search(
        query_embedding,
        top_k=top_k
    )

    # Vector similarity voting
    votes = {}

    for r in results:

        label = (
            r["id"]
            .split('-')[0]
            .split('_')[0]
        )

        votes[label] = (
            votes.get(label, 0)
            + r["score"]
        )

    predicted_label = (
        max(votes, key=votes.get)
        if votes
        else "Unknown"
    )

    # Qwen reasoning
    if use_llm:

        llm_prediction = llm_reasoning(
            test_doc["content"],
            results
        )

        return (
            llm_prediction,
            [r["id"] for r in results]
        )

    return (
        predicted_label,
        [r["id"] for r in results]
    )


##############################################
# 8. Evaluation
##############################################

def evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
):

    test_docs = load_and_process_docs(
        test_folder
    )

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = (
            doc["id"]
            .split('-')[0]
            .split('_')[0]
        )

        total_counts[true_label] += 1

        predicted_label, _ = detect_pattern(
            {"content": doc["raw"]},
            vector_store,
            embedder,
            top_k=top_k
        )

        if (
            predicted_label.lower()
            == true_label.lower()
        ):
            correct_counts[true_label] += 1

        print(
            f"{doc['id']} | "
            f"True: {true_label} | "
            f"Pred: {predicted_label}"
        )

    # Metrics
    per_pattern_accuracy = {
        label:
            correct_counts[label]
            / total_counts[label]
        for label in total_counts
    }

    overall_accuracy = (
        sum(correct_counts.values())
        / sum(total_counts.values())
    )

    print("\nPer-Pattern Accuracy:")

    for label, acc in per_pattern_accuracy.items():
        print(
            f"{label}: {acc:.4f}"
        )

    print(
        f"\nOverall Accuracy: "
        f"{overall_accuracy:.4f}"
    )

    return (
        overall_accuracy,
        per_pattern_accuracy
    )



/volatile/home/ya279835/Projets/perso/P-MART-RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===============================
# 0. Set your OpenAI API Key
# ===============================


# ===============================
# 1. Build the RAG index
# ===============================
data_folder = "PMART_PUML/knowledge_base"
vector_store, embedder = build_rag_index(data_folder)



# ===============================
# 2. Evaluate accuracy on a test folder
# ===============================
test_folder = "PMART_PUML/puml_test/singleton"
overall_acc, per_pattern_acc = evaluate_detection(
    test_folder,
    vector_store,
    embedder,
    top_k=5
)

Loading PlantUML files...


  1%|          | 1/93 [00:20<32:07, 20.95s/it]

nons_56.puml | True: nons | Pred: DrawingView


  2%|▏         | 2/93 [00:43<33:36, 22.16s/it]

singleton_21.puml | True: singleton | Pred: nons


...

nons_36.puml | True: nons | Pred: nons


 19%|█▉        | 18/93 [10:52<33:01, 26.42s/it]

nons_51.puml | True: nons | Pred: Singleton


 19%|█▉        | 18/93 [11:05<46:11, 36.95s/it]


KeyboardInterrupt: 